# Guardian JSON -> FinBERT -> PCA32 (Colab)

This notebook is designed for **easy Colab operation**:
1. Upload one Guardian JSON file (e.g., `guardian_news_2022.json`)
2. Set config (year, output prefix, PCA dims)
3. Run all cells
4. Download parquet outputs

Outputs:
- Daily `(ticker, date)` FinBERT features with 768-d embeddings
- PCA-compressed features (default 32 dims)
- Optional article-level FinBERT output


In [ ]:
# Install dependencies (Colab)
!pip -q install pyarrow transformers torch scikit-learn pandas_market_calendars tqdm

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas_market_calendars as mcal

from google.colab import files

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# ===== User config =====
TARGET_YEAR = 2022
OUTPUT_PREFIX = f'news_finbert_{TARGET_YEAR}'
PCA_COMPONENTS = 32   # set 32 or 64
BATCH_SIZE = 32
MAX_LENGTH = 512

# If True, also save article-level FinBERT output parquet
SAVE_ARTICLE_LEVEL = True

# Optional: set explicit file name if you upload multiple files
JSON_FILE_NAME = None  # e.g., 'guardian_news_2022.json'


In [ ]:
# Upload JSON
uploaded = files.upload()
uploaded_names = list(uploaded.keys())
if not uploaded_names:
    raise ValueError('No file uploaded.')

if JSON_FILE_NAME is not None:
    if JSON_FILE_NAME not in uploaded_names:
        raise ValueError(f'{JSON_FILE_NAME} not found in uploaded files: {uploaded_names}')
    json_path = Path(JSON_FILE_NAME)
else:
    json_path = Path(uploaded_names[0])

print('Using JSON file:', json_path)

with open(json_path, 'r', encoding='utf-8') as f:
    articles = json.load(f)

df = pd.DataFrame(articles)
print('Loaded rows:', len(df))
print('Columns:', df.columns.tolist())

if 'publication_date' not in df.columns:
    raise ValueError('Expected `publication_date` in JSON rows.')

df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
df = df[df['publication_date'].dt.year == TARGET_YEAR].copy()
print(f'Rows after year={TARGET_YEAR} filter:', len(df))
if df.empty:
    raise ValueError('No rows remain after year filter.')

In [ ]:
# Build text input and trading date mapping
def make_text(row):
    preferred = row.get('text_for_finbert', None)
    if isinstance(preferred, str) and preferred.strip():
        return preferred.strip()
    parts = [
        row.get('headline', ''),
        row.get('trailText', ''),
        row.get('bodyText', ''),
    ]
    text = ' '.join([p for p in parts if isinstance(p, str) and p.strip()])
    return text.strip()

df['text_input'] = df.apply(make_text, axis=1)
df = df[df['text_input'].str.len() > 0].copy()

# trading_date in your JSON is often null, so fallback to publication_date.
if 'trading_date' in df.columns:
    td = pd.to_datetime(df['trading_date'], errors='coerce')
else:
    td = pd.Series(pd.NaT, index=df.index)

df['event_date'] = td.fillna(df['publication_date'])

# Shift weekends to next business day; then align to NYSE schedule.
df['event_date'] = pd.to_datetime(df['event_date']).dt.normalize()
weekend = df['event_date'].dt.dayofweek >= 5
df.loc[weekend, 'event_date'] = df.loc[weekend, 'event_date'] + pd.offsets.BDay(1)

print('Rows ready for FinBERT:', len(df))

In [ ]:
# FinBERT inference: 3-way probs + 768-d [CLS] embedding
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'ProsusAI/finbert'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True)
model = model.to(device)
model.eval()

texts = df['text_input'].tolist()
all_probs = []
all_emb = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='FinBERT batches'):
        batch_texts = texts[i:i+BATCH_SIZE]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        out = model(**enc)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        # Last hidden state: [batch, seq_len, 768], take CLS token
        cls = out.hidden_states[-1][:, 0, :].cpu().numpy()

        all_probs.append(probs)
        all_emb.append(cls)

probs = np.vstack(all_probs)
emb = np.vstack(all_emb)

# ProsusAI/finbert label order is usually [positive, negative, neutral].
# We map by label2id to avoid assumptions.
id2label = model.config.id2label
label2idx = {v.lower(): int(k) for k, v in id2label.items()}
required = ['positive', 'negative', 'neutral']
for r in required:
    if r not in label2idx:
        raise ValueError(f'Label {r} not found in model labels: {label2idx}')

df['sentiment_pos'] = probs[:, label2idx['positive']]
df['sentiment_neg'] = probs[:, label2idx['negative']]
df['sentiment_neu'] = probs[:, label2idx['neutral']]

for j in range(emb.shape[1]):
    df[f'emb_{j}'] = emb[:, j].astype(np.float32)

print('Article-level inference complete.')
print(df[['sentiment_pos','sentiment_neg','sentiment_neu']].head())

In [ ]:
# Aggregate to daily (ticker, date), then create full ticker-date grid with has_news flag
df['date'] = pd.to_datetime(df['event_date']).dt.normalize()

emb_cols = [f'emb_{i}' for i in range(768)]
agg_cols = ['sentiment_pos', 'sentiment_neu', 'sentiment_neg'] + emb_cols

daily_news = (
    df.groupby(['ticker', 'date'], as_index=False)
      .agg({**{c: 'mean' for c in agg_cols}, 'text_input': 'count'})
      .rename(columns={'text_input': 'n_articles'})
)
daily_news['has_news'] = 1
daily_news['mean_tone'] = daily_news['sentiment_pos'] - daily_news['sentiment_neg']

# Full NYSE date grid for the year and observed tickers in uploaded file.
nyse = mcal.get_calendar('NYSE')
sched = nyse.schedule(start_date=f'{TARGET_YEAR}-01-01', end_date=f'{TARGET_YEAR}-12-31')
trading_days = pd.to_datetime(sched.index).normalize()
tickers = sorted(df['ticker'].dropna().unique().tolist())

grid = pd.MultiIndex.from_product([tickers, trading_days], names=['ticker', 'date']).to_frame(index=False)
daily = grid.merge(daily_news, on=['ticker', 'date'], how='left')

fill_zero_cols = agg_cols + ['n_articles', 'has_news', 'mean_tone']
for c in fill_zero_cols:
    daily[c] = daily[c].fillna(0.0)

daily['n_articles'] = daily['n_articles'].astype(np.int32)
daily['has_news'] = daily['has_news'].astype(np.int8)

print('Daily rows:', len(daily))
print('Tickers:', len(tickers), 'Trading days:', len(trading_days))
print('News-day share:', float((daily['has_news'] == 1).mean()))

In [ ]:
# PCA compression to 32/64 dims (fit on news days only)
if PCA_COMPONENTS not in [32, 64]:
    raise ValueError('PCA_COMPONENTS should be 32 or 64 for this workflow.')

fit_mask = daily['has_news'] == 1
if fit_mask.sum() < PCA_COMPONENTS:
    raise ValueError(f'Not enough news rows ({fit_mask.sum()}) for PCA-{PCA_COMPONENTS}.')

X_fit = daily.loc[fit_mask, emb_cols].to_numpy(dtype=np.float64)
X_all = daily[emb_cols].to_numpy(dtype=np.float64)

pca = PCA(n_components=PCA_COMPONENTS, svd_solver='full', random_state=42)
pca.fit(X_fit)
Z = pca.transform(X_all)

pca_cols = [f'news_pca_{i:02d}' for i in range(PCA_COMPONENTS)]
pca_df = pd.DataFrame(Z, columns=pca_cols)

daily_pca = pd.concat([
    daily[['ticker', 'date', 'sentiment_pos', 'sentiment_neu', 'sentiment_neg', 'n_articles', 'has_news', 'mean_tone']].reset_index(drop=True),
    pca_df.reset_index(drop=True)
], axis=1)

print(f'PCA-{PCA_COMPONENTS} explained variance: {pca.explained_variance_ratio_.sum():.4f}')

In [ ]:
# Save parquet outputs and download
out_daily_768 = Path(f'{OUTPUT_PREFIX}_daily_768.parquet')
out_daily_pca = Path(f'{OUTPUT_PREFIX}_daily_pca{PCA_COMPONENTS}.parquet')
out_article = Path(f'{OUTPUT_PREFIX}_articles_finbert.parquet')

daily.to_parquet(out_daily_768, index=False)
daily_pca.to_parquet(out_daily_pca, index=False)

print('Saved:', out_daily_768)
print('Saved:', out_daily_pca)

if SAVE_ARTICLE_LEVEL:
    keep_article_cols = [
        'ticker', 'publication_date', 'event_date', 'date',
        'sentiment_pos', 'sentiment_neu', 'sentiment_neg',
        'text_input'
    ] + emb_cols
    df[keep_article_cols].to_parquet(out_article, index=False)
    print('Saved:', out_article)

# Download files to local machine
files.download(str(out_daily_768))
files.download(str(out_daily_pca))
if SAVE_ARTICLE_LEVEL:
    files.download(str(out_article))